In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

import json

# Настройки визуализации
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (8,6)

# Папки
DATA_DIR = "data"
ARTIFACTS_DIR = "artifacts"
FIGURES_DIR = os.path.join(ARTIFACTS_DIR, "figures")
LABELS_DIR = os.path.join(ARTIFACTS_DIR, "labels")

for folder in [ARTIFACTS_DIR, FIGURES_DIR, LABELS_DIR]:
    os.makedirs(folder, exist_ok=True)

In [2]:
# Список выбранных CSV
datasets = [
    "S07-hw-dataset-01.csv",
    "S07-hw-dataset-02.csv",
    "S07-hw-dataset-03.csv"
]

In [4]:
# Словари для артефактов
metrics_summary = {}
best_configs = {}

In [6]:
# Функция для EDA и препроцессинга
def load_and_preprocess(file_path):
    df = pd.read_csv(file_path)
    print(f"\n=== Dataset: {file_path} ===")
    display(df.head())
    print(df.info())
    print(df.describe())
    print("Пропуски по колонкам:")
    print(df.isna().sum())

    # Разделяем sample_id и признаки
    if "sample_id" in df.columns:
        sample_id = df["sample_id"]
        X = df.drop(columns=["sample_id"])
    else:
        sample_id = pd.Series(np.arange(len(df)), name="sample_id")
        X = df.copy()

    # Определяем числовые и категориальные признаки
    numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

    # Препроцессинг pipeline
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    if categorical_features:
        categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ])
        preprocessor = ColumnTransformer(transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ])
    else:
        preprocessor = ColumnTransformer(transformers=[
            ('num', numeric_transformer, numeric_features)
        ])

    X_scaled = preprocessor.fit_transform(X)
    return X_scaled, sample_id, X, preprocessor

In [7]:
# Функции для метрик
def compute_metrics(X, labels):
    # silhouette_score нельзя считать для одного кластера
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    if n_clusters <= 1:
        return {"silhouette": np.nan, "davies_bouldin": np.nan, "calinski_harabasz": np.nan}

    # Для DBSCAN учитываем шум
    non_noise_idx = np.where(labels != -1)[0]
    X_non_noise = X[non_noise_idx]
    labels_non_noise = labels[non_noise_idx]

    silhouette = silhouette_score(X_non_noise, labels_non_noise)
    db = davies_bouldin_score(X_non_noise, labels_non_noise)
    ch = calinski_harabasz_score(X_non_noise, labels_non_noise)
    return {"silhouette": silhouette, "davies_bouldin": db, "calinski_harabasz": ch}

In [9]:
# Основной цикл по датасетам
for i, dataset_file in enumerate(datasets, start=1):
    X_scaled, sample_id, X_raw, preprocessor = load_and_preprocess(os.path.join(DATA_DIR, dataset_file))

    # 6.1 KMeans: подбор k по silhouette
    silhouette_scores = []
    k_values = range(2, 11)
    for k in k_values:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels_km = km.fit_predict(X_scaled)
        score = silhouette_score(X_scaled, labels_km)
        silhouette_scores.append(score)

    # Лучший k
    best_k = k_values[np.argmax(silhouette_scores)]
    km_best = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    labels_km_best = km_best.fit_predict(X_scaled)

    # График silhouette vs k
    plt.figure()
    plt.plot(k_values, silhouette_scores, marker='o')
    plt.title(f"KMeans silhouette vs k - Dataset {i}")
    plt.xlabel("k")
    plt.ylabel("Silhouette Score")
    plt.grid(True)
    fig_path = os.path.join(FIGURES_DIR, f"silhouette_vs_k_ds{i}.png")
    plt.savefig(fig_path)
    plt.close()

    # Метрики KMeans
    metrics_km = compute_metrics(X_scaled, labels_km_best)

    # 6.2 DBSCAN: простой подбор eps
    # В реальной работе eps подбирают через k-distance plot; здесь примерный перебор
    eps_values = np.linspace(0.5, 3.0, 6)
    best_silhouette = -1
    best_dbscan = None
    best_labels_db = None
    for eps in eps_values:
        dbscan = DBSCAN(eps=eps, min_samples=5)
        labels_db = dbscan.fit_predict(X_scaled)
        # пропускаем если все точки шум
        if len(set(labels_db)) <= 1:
            continue
        metrics_db_temp = compute_metrics(X_scaled, labels_db)
        if metrics_db_temp["silhouette"] > best_silhouette:
            best_silhouette = metrics_db_temp["silhouette"]
            best_dbscan = dbscan
            best_labels_db = labels_db

    if best_labels_db is not None:
        metrics_db = compute_metrics(X_scaled, best_labels_db)
        labels_to_save = best_labels_db
        best_method = "DBSCAN" if metrics_db["silhouette"] > metrics_km["silhouette"] else "KMeans"
    else:
        metrics_db = None
        labels_to_save = labels_km_best
        best_method = "KMeans"

    # PCA 2D визуализация для лучшего решения
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X_scaled)
    plt.figure()
    plt.scatter(X_pca[:,0], X_pca[:,1], c=labels_to_save, cmap='tab10', s=50)
    plt.title(f"PCA(2D) - Dataset {i} - {best_method}")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.colorbar(label="Cluster")
    pca_fig_path = os.path.join(FIGURES_DIR, f"pca_ds{i}.png")
    plt.savefig(pca_fig_path)
    plt.close()

    # Сохраняем метки
    labels_df = pd.DataFrame({"sample_id": sample_id, "cluster_label": labels_to_save})
    labels_df.to_csv(os.path.join(LABELS_DIR, f"labels_hw07_ds{i}.csv"), index=False)

    # Сохраняем метрики
    metrics_summary[f"dataset_{i}"] = {
        "KMeans": metrics_km,
        "DBSCAN": metrics_db if metrics_db else "DBSCAN failed",
        "best_method": best_method
    }

    # Лучшие параметры
    best_configs[f"dataset_{i}"] = {
        "best_method": best_method,
        "KMeans_k": best_k,
        "DBSCAN_eps": best_dbscan.eps if best_dbscan else None
    }


=== Dataset: data/S07-hw-dataset-01.csv ===


,sample_id,f01,f02,f03,f04,f05,f06,f07,f08
0,0,-0.536647,-69.812900,-0.002657,71.743147,-11.396498,-12.291287,-6.836847,-0.504094
1,1,15.230731,52.727216,-1.273634,-104.123302,11.589643,34.316967,-49.468873,0.390356
2,2,18.542693,77.317150,-1.321686,-111.946636,10.254346,25.892951,44.595250,0.325893
3,3,-12.538905,-41.709458,0.146474,16.322124,1.391137,2.014316,-39.930582,0.139297
4,4,-6.903056,61.833444,-0.022466,-42.631335,3.107154,-5.471054,7.001149,0.131213


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sample_id  12000 non-null  int64  
 1   f01        12000 non-null  float64
 2   f02        12000 non-null  float64
 3   f03        12000 non-null  float64
 4   f04        12000 non-null  float64
 5   f05        12000 non-null  float64
 6   f06        12000 non-null  float64
 7   f07        12000 non-null  float64
 8   f08        12000 non-null  float64
dtypes: float64(8), int64(1)
memory usage: 843.9 KB
None
         sample_id           f01           f02           f03           f04  \
count  12000.00000  12000.000000  12000.000000  12000.000000  12000.000000   
mean    5999.50000     -2.424716     19.107804     -0.222063     -8.284501   
std     3464.24595     11.014315     60.790338      0.500630     59.269838   
min        0.00000    -19.912573    -92.892652     -1.590979   -134.303679   
25%  

,sample_id,x1,x2,z_noise
0,0,0.098849,-1.846034,21.288122
1,1,-1.024516,1.829616,6.072952
2,2,-1.094178,-0.158545,-18.938342
3,3,-1.612808,-1.565844,-11.629462
4,4,1.659901,-2.133292,1.895472


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sample_id  8000 non-null   int64  
 1   x1         8000 non-null   float64
 2   x2         8000 non-null   float64
 3   z_noise    8000 non-null   float64
dtypes: float64(3), int64(1)
memory usage: 250.1 KB
None
        sample_id           x1           x2      z_noise
count  8000.00000  8000.000000  8000.000000  8000.000000
mean   3999.50000     0.478867     0.241112     0.110454
std    2309.54541     0.955138     0.663195     8.097716
min       0.00000    -2.487352    -2.499237   -34.056074
25%    1999.75000    -0.116516    -0.242357    -5.392210
50%    3999.50000     0.490658     0.241092     0.132470
75%    5999.25000     1.085263     0.726526     5.655605
max    7999.00000     2.987555     2.995553    29.460076
Пропуски по колонкам:
sample_id    0
x1           0
x2           0
z_noise      0
dt

,sample_id,x1,x2,f_corr,f_noise
0,0,-2.710470,4.997107,-1.015703,0.718508
1,1,8.730238,-8.787416,3.953063,-1.105349
2,2,-1.079600,-2.558708,0.976628,-3.605776
3,3,6.854042,1.560181,1.760614,-1.230946
4,4,9.963812,-8.869921,2.966583,0.915899


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sample_id  15000 non-null  int64  
 1   x1         15000 non-null  float64
 2   x2         15000 non-null  float64
 3   f_corr     15000 non-null  float64
 4   f_noise    15000 non-null  float64
dtypes: float64(4), int64(1)
memory usage: 586.1 KB
None
          sample_id            x1            x2        f_corr       f_noise
count  15000.000000  15000.000000  15000.000000  15000.000000  15000.000000
mean    7499.500000      1.246296      1.033764      0.212776     -0.027067
std     4330.271354      4.592421      4.710791      1.530017      2.506375
min        0.000000     -9.995585     -9.980853     -5.212038     -8.785884
25%     3749.750000     -1.782144     -2.666393     -0.966224     -1.731128
50%     7499.500000      0.664226      1.831257      0.296508     -0.052391
75%    11249.250000    

In [10]:
# Проверка устойчивости KMeans на Dataset 1
X_scaled1, sample_id1, X_raw1, preprocessor1 = load_and_preprocess(os.path.join(DATA_DIR, datasets[0]))
ari_scores = []
from sklearn.metrics import adjusted_rand_score
labels_list = []
for seed in [0, 42, 100, 200, 999]:
    km = KMeans(n_clusters=best_configs["dataset_1"]["KMeans_k"], n_init=10, random_state=seed)
    labels = km.fit_predict(X_scaled1)
    labels_list.append(labels)
# ARI между парой разбиений
for i1 in range(len(labels_list)):
    for i2 in range(i1+1, len(labels_list)):
        ari = adjusted_rand_score(labels_list[i1], labels_list[i2])
        ari_scores.append(ari)
print(f"Dataset 1 KMeans stability (ARI) across 5 seeds: mean={np.mean(ari_scores):.3f}, min={np.min(ari_scores):.3f}")


=== Dataset: data/S07-hw-dataset-01.csv ===


,sample_id,f01,f02,f03,f04,f05,f06,f07,f08
0,0,-0.536647,-69.812900,-0.002657,71.743147,-11.396498,-12.291287,-6.836847,-0.504094
1,1,15.230731,52.727216,-1.273634,-104.123302,11.589643,34.316967,-49.468873,0.390356
2,2,18.542693,77.317150,-1.321686,-111.946636,10.254346,25.892951,44.595250,0.325893
3,3,-12.538905,-41.709458,0.146474,16.322124,1.391137,2.014316,-39.930582,0.139297
4,4,-6.903056,61.833444,-0.022466,-42.631335,3.107154,-5.471054,7.001149,0.131213


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sample_id  12000 non-null  int64  
 1   f01        12000 non-null  float64
 2   f02        12000 non-null  float64
 3   f03        12000 non-null  float64
 4   f04        12000 non-null  float64
 5   f05        12000 non-null  float64
 6   f06        12000 non-null  float64
 7   f07        12000 non-null  float64
 8   f08        12000 non-null  float64
dtypes: float64(8), int64(1)
memory usage: 843.9 KB
None
         sample_id           f01           f02           f03           f04  \
count  12000.00000  12000.000000  12000.000000  12000.000000  12000.000000   
mean    5999.50000     -2.424716     19.107804     -0.222063     -8.284501   
std     3464.24595     11.014315     60.790338      0.500630     59.269838   
min        0.00000    -19.912573    -92.892652     -1.590979   -134.303679   
25%  

In [11]:
# Сохраняем артефакты
with open(os.path.join(ARTIFACTS_DIR, "metrics_summary.json"), "w") as f:
    json.dump(metrics_summary, f, indent=4)
with open(os.path.join(ARTIFACTS_DIR, "best_configs.json"), "w") as f:
    json.dump(best_configs, f, indent=4)

print("Все артефакты сохранены в папке 'artifacts/'.")

Все артефакты сохранены в папке 'artifacts/'.
